# Lab: Build an Agentic Travel Planner

You will build a LangGraph agent that

1. **asks you questions** until it knows the destination, the dates and your interests (*human-in-the-loop*),
2. spawns **one research sub-agent per interest** in parallel (*map-reduce with `Send()`*),
3. builds a **day-by-day schedule**, which a **critic agent** reviews (*structured output + loops*),
4. renders a well structured **`travel_plan.html`**.

| Milestone | What you build | LangGraph concept |
|---|---|---|
| M1 | Intake loop | structured output, `interrupt()`, conditional edges, checkpointer |
| M2 | Interest research | `Send()`, sub-graphs, reducers, Tavily + Wikipedia |
| M3 | Geo data + scheduling | OpenStreetMap, structured output |
| M4 | Critique loop | cycles, human approval |
| M5 | HTML output | final artefact |


In [ ]:
import base64
from IPython.display import Image, display

def showmm(graph):
    encoded = base64.urlsafe_b64encode(graph.encode()).decode()
    url = f"https://mermaid.ink/img/{encoded}"
    display(Image(url=url))

## Setup

Create a `.env` file next to this notebook with `OPENAI_API_KEY` and `TAVILY_API_KEY`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    assert os.environ.get(key), f"Missing {key} in .env"

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# M1 — Intake: ask until the request is complete

The user types free text such as *"I want to go to Lisbon for 3 days in March, I like food, architecture and the sea"*.
We need **three** things: a destination, concrete dates and at least one interest.

Plan for the graph:

In [ ]:
graph = """flowchart TD
    START([Start]) --> EXTRACT[Extract trip details]
    EXTRACT --> VALIDATE{Request complete?}

    VALIDATE -->|Yes| END([End])
    VALIDATE -->|No| ASK[Ask for missing details]
    ASK --> PAUSE[[Wait for human response]]
    PAUSE --> EXTRACT
"""

showmm(graph)



### Concepts
* **Structured output**: `llm.with_structured_output(Schema)` forces the LLM to return a Pydantic object.
* **`interrupt()`**: pauses the graph and hands a question to the human. The graph is resumed with `Command(resume=answer)`. This requires a **checkpointer**, which saves the state while the graph is paused.

### 📚 Resources — M1

**LangGraph**
* [Graph API: state, nodes, edges, reducers](https://docs.langchain.com/oss/python/langgraph/graph-api) — `StateGraph`, `START`/`END`, conditional edges
* [How-to: use the Graph API](https://docs.langchain.com/oss/python/langgraph/use-graph-api) — state schemas, input/output schemas, reducers in practice
* [Interrupts (human-in-the-loop)](https://docs.langchain.com/oss/python/langgraph/interrupts) — `interrupt()` and `Command(resume=...)`
* [Persistence: checkpointers and threads](https://docs.langchain.com/oss/python/langgraph/persistence) — why `interrupt()` needs a checkpointer and a `thread_id`

**Other**
* [Structured output in LangChain](https://docs.langchain.com/oss/python/langchain/structured-output) — `with_structured_output(...)`
* [Pydantic: models and fields](https://docs.pydantic.dev/latest/concepts/models/) — `BaseModel`, `Field`, `Optional`

Next, we define the graph state for the travel agent. It will hold the destination, dates and interests, as well as a flag to indicate whether the request is complete. Our models will be Pydantic models, which are used for structured output in LangChain.

We want:
- `destination`
- `start_date` and `end_date`
- `interests`: this is a list -> we will spawn one research sub-agent per interest in parallel

Using a pydantic model for the request allows us to use structured output in LangChain, which makes it easier to extract the relevant information from the user's input.

In [ ]:
from datetime import date
from typing import Optional
from pydantic import BaseModel, Field


class TripRequest(BaseModel):
    """Everything we need to know to plan a trip. Unknown fields stay None / empty."""
    destination: Optional[str] = Field(None, description="City or region to visit, e.g. 'Lisbon, Portugal'.")
    start_date: Optional[date] = Field(None, description="First day of the trip (ISO date).")
    end_date: Optional[date] = Field(None, description="Last day of the trip (ISO date).")
    interests: list[str] = Field(default_factory=list, description="Short interests, e.g. ['food', 'street art'].")

### State

This is what the graph will hold in its state:

- `messages` uses the `add_messages` reducer, so every node appends to the conversation. Otherwise, each node would overwrite the previous messages.
- `request` is overwritten each time we re-extract.
- `problems` lists what is still wrong.

Remember that the graph nodes will be executed in order, and the state is passed from one node to the next. The graph will start with an empty state, and each node will update the state as it goes along.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages

MAX_TRIP_DAYS = 7


class TripState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    request: dict  # a TripRequest dumped to JSON types, so the checkpointer can serialise it
    problems: list[str]

### Nodes

We will have three nodes:

- `extract` reads the *whole* conversation and re-builds the `TripRequest`. Today's date is part of the prompt so that
"next Friday" or "in March" can be resolved.
- `validate` is plain Python (no LLM needed!) and returns a list of problems.
- `ask` is an LLM node that asks the user for the missing information. It uses `interrupt()` to pause the graph and wait for the human to answer.

If `validate` returns an empty list, the graph is done. Otherwise, it loops back to `extract`.

In [ ]:
from langgraph.graph import StateGraph, START, END

def demo_extract(state: TripState):
    print("extract: re-build the TripRequest from the conversation")
    return {}


def demo_validate(state: TripState):
    print("validate: check the request, return a list of problems")
    return {"problems": state.get("problems", [])}


def demo_ask(state: TripState):
    print("ask: formulate a question and pause with interrupt()")
    return {}


def demo_route(state: TripState):
    return "ask" if state.get("problems") else END


demo_builder = StateGraph(TripState)
demo_builder.add_node("extract", demo_extract)
demo_builder.add_node("validate", demo_validate)
demo_builder.add_node("ask", demo_ask)

demo_builder.add_edge(START, "extract")
demo_builder.add_edge("extract", "validate")
demo_builder.add_conditional_edges("validate", demo_route, ["ask", END])
demo_builder.add_edge("ask", "extract")

demo_graph = demo_builder.compile()
display(Image(demo_graph.get_graph().draw_mermaid_png()))

In [ ]:
# run the demo graph with an empty state, so we can see the print statements
# TODO: add an non-empty problems list to demo_validate(). What will happen?
demo_graph.invoke({"messages": [], "request": {}, "problems": []})

In [ ]:
extract_instructions = """You extract trip details from a conversation with a traveller.
Today is {today}.

Read the WHOLE conversation and return the up-to-date trip request.
- Only fill a field if the traveller stated it (or it follows directly from what they said, e.g. '3 days from 12 March').
- Resolve relative dates ("next Friday") using today's date. If only a month is given, leave the dates empty.
- Interests are short phrases. Split "food and architecture" into two interests.
- Never invent information. Leave unknown fields empty."""

# we want the llm to return a TripRequest, so we use structured output
extractor = llm.with_structured_output(TripRequest)


def extract(state: TripState):
    system = SystemMessage(content=extract_instructions.format(today=date.today().isoformat()))
    request = extractor.invoke([system] + state["messages"])
    return {"request": request.model_dump(mode="json")}

In [ ]:
# TODO: check the model works as expected
# create an instance of TripState with a sample message and call the extract function
# as a user message, use "next Friday" or "in March" to test relative date resolution:
# how does the model handle it? How can you improve the prompt to get the desired behaviour?


<details>
<summary>🔍 Click here to see a solution</summary>

<details>
<summary> Click here if you really need to </summary>

Running the extract function:
```python
ts = TripState(messages=[HumanMessage(content="I want to go to Lisbon for 3 days in March, I like food and architecture")], request={}, problems=[])
res = extract(ts)
res
```

If you run the above code several times, you will see that the the model often assumes a specific date in March, which is not what we want. We want the model to leave the dates empty if it cannot resolve them. You can improve the prompt by adding a sentence like "If only a month is given, leave the dates empty." to the prompt template.
</details>
</details>

### The validator

The solution below checks all the rules. Try to write it yourself first: it should return a list of
human readable problems (an empty list means the request is complete). Rules:

* destination missing
* start or end date missing
* end date before start date, or start date in the past
* trip longer than `MAX_TRIP_DAYS`
* no interests

In [ ]:
# TODO: in the validate function, we collect problems with the request, i.e. if a field is missing or invalid, we append a string to the problems list.
# For example, if the destination is missing, we append "Missing destination". We have slot `problems` in the state, which is a list of strings.
# If the request is valid, we return an empty list.

# replace the `pass` statements and test the code
def validate(state: TripState):
    r = TripRequest(**state["request"])
    problems = []
    if not r.destination:
        pass
    if not r.start_date or not r.end_date:
        pass
    else:
        if r.end_date < r.start_date:
            pass
        elif r.start_date < date.today():
            pass
        elif (r.end_date - r.start_date).days + 1 > MAX_TRIP_DAYS:
            pass
    if not r.interests:
        pass
    return {"problems": problems}


In [ ]:
# Use the checked-in implementation by default. Set this to False after implementing validate above.
USE_REFERENCE_SOLUTION = True
if USE_REFERENCE_SOLUTION:
    from travel_agent_support.solutions import validate as _validate_solution

    def validate(state: TripState):
        return _validate_solution(state, max_trip_days=MAX_TRIP_DAYS)


<details>
<summary>Reference solution</summary>

The runnable reference implementation is in `travel_agent_support/solutions.py` and is loaded by default in the code cell above. Set `USE_REFERENCE_SOLUTION = False` after implementing the exercise to run your own validator.
</details>

`ask` is the human-in-the-loop node. It asks the LLM to formulate **one friendly question** covering all open problems,
then calls `interrupt(question)`. When the graph is resumed, `interrupt()` *returns* the human's answer, which we append as a `HumanMessage`.

In [ ]:
from langgraph.types import interrupt, Command

ask_instructions = """You are a friendly travel agent collecting trip details.
Known so far: {known}
Open problems: {problems}
Write ONE short, friendly message that asks the traveller to fix/provide exactly these open points. Do not ask about anything else."""


def ask(state: TripState):
    prompt = ask_instructions.format(
        known=state["request"],
        problems="; ".join(state["problems"]),
    )
    question = llm.invoke(prompt).content
    answer = interrupt(question)  # graph pauses here until Command(resume=...)
    return {"messages": [AIMessage(content=question), HumanMessage(content=answer)]}

Testing:
 - the agent asks a question and you can answer it (you can spot the answer in the `messages` list)
 - we haven't yet implemented the loop, so the graph will stop after one question. You can see the `problems` list in the state, which is still non-empty

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

ts = TripState({
    "messages": [],
    "request": {"destination": "Lisbon, Portugal"},
    "problems": ["the exact start and end dates are missing", "no interests were given"]
})

config = {"configurable": {"thread_id": "ask-test-1"}}

ask_test_builder = StateGraph(TripState)
ask_test_builder.add_node("ask", ask)
ask_test_builder.add_edge(START, "ask")
ask_test_graph = ask_test_builder.compile(checkpointer=MemorySaver())

# Run until ask() calls interrupt()
result = ask_test_graph.invoke(ts, config=config)
question = result["__interrupt__"][0].value
print(f"Agent asks: {question}")

# Type your reply, then resume the paused graph
answer = input("Your answer: ")
result = ask_test_graph.invoke(Command(resume=answer), config=config)

result

Next, we have to terminate the graph when all problems are solved. We do this by adding a [conditional edge](https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges) from `validate` to `END` if the list of problems is empty. In this node, there is no LLM involved.

In [ ]:
def route_after_validate(state: TripState):
    # we go to the ask node if there are problems, otherwise we go to END
    # you have to name name the ask node "ask", i.e. the string "ask" must match the name of the node in the graph
    return "ask" if state["problems"] else END

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

interview_builder = StateGraph(TripState)
interview_builder.add_node("extract", extract)
interview_builder.add_node("validate", validate)
interview_builder.add_node("ask", ask) # this has to be named "ask" so the route_after_validate function can find it

interview_builder.add_edge(START, "extract")
interview_builder.add_edge("extract", "validate")
interview_builder.add_conditional_edges("validate", route_after_validate, ["ask", END])
interview_builder.add_edge("ask", "extract")

# a checkpointer is needed to store the state of the graph between invocations, so we can pause and resume the graph.
interview_graph = interview_builder.compile(checkpointer=MemorySaver())

In [ ]:
from IPython.display import Image, display

display(Image(interview_graph.get_graph().draw_mermaid_png()))

### Running a graph that pauses

`invoke` returns early with an `__interrupt__` entry when the graph hits `interrupt()`. The helper below shows the
question, reads the answer with `input()` and resumes with `Command(resume=answer)` until the graph is done.

For automated runs you can pass `scripted_answers`.

In [ ]:
def run_with_human(graph, graph_input, config, scripted_answers=None):
    scripted_answers = list(scripted_answers or [])
    result = graph.invoke(graph_input, config)
    while "__interrupt__" in result:
        question = result["__interrupt__"][0].value
        print(f"\n🤖 {question}")
        if scripted_answers:
            answer = scripted_answers.pop(0)
            print(f"👤 {answer}")
        else:
            answer = input("👤 ")
        result = graph.invoke(Command(resume=answer), config)
    return result

In [ ]:
# with thread_id, we can run multiple interviews in parallel, and the checkpointer will keep the state of each interview separate.
config = {"configurable": {"thread_id": "intake-1"}}
first_message = {"messages": [HumanMessage(content="I want to go to Lisbon and I love food and architecture.")]}

# Leave scripted_answers out to type the answers yourself.
result = run_with_human(
    interview_graph, first_message, config,
    scripted_answers=["From the 14th to the 16th of next month"],
)
TripRequest(**result["request"])

In [ ]:
result

**M1 done.** The agent keeps asking until `validate` returns no problems.

# M2 — Research: one sub-agent per interest (map-reduce)

The number of interests is only known at runtime (e.g. *"food, architecture, sea"* = 3, but it could be 5), so we cannot draw the edges in advance.
This is exactly what **`Send()`** is for (see also module 4 in https://academy.langchain.com/courses/intro-to-langgraph):

In [ ]:
graph = """flowchart TD
    VALIDATE[validate] --> FOOD[research_interest: food]
    VALIDATE --> ARCH[research_interest: architecture]
    VALIDATE --> SEA[research_interest: sea]

    FOOD --> REDUCE[(activities reducer)]
    ARCH --> REDUCE
    SEA --> REDUCE
    """
showmm(graph)

In a nutshell:
* **map**: a conditional edge that is called like this `Send("research_interest", {interest_1, ..., interest_N})`. Send starts $N$ sub-agents in parallel (`research_interest(interest_i)` for each interest).
* **reduce**: combines the results of all sub-agents. In our case, every sub-agent returns `activities`; the **`operator.add` reducer** concatenates the lists.

Each sub-agent is itself a small graph (a *sub-graph*), for each interest we run the following graph:

In [ ]:
graph = """flowchart TD
    START([START]) --> WRITE[write_queries]
    WRITE --> WEB[search_web: Tavily]
    WRITE --> WIKI[search_wikipedia]
    WEB --> EXTRACT[extract_activities]
    WIKI --> EXTRACT
    EXTRACT --> END([END])
    """
showmm(graph)

### 📚 Resources — M2

**LangGraph**
* [Graph API: `Send` (map-reduce) and reducers](https://docs.langchain.com/oss/python/langgraph/graph-api#send) — see the sections on `Send` and on reducers
* [Subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs) — a compiled graph used as a node
* [Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents) — parallelization, orchestrator-worker and ReAct-style agents

**Search tools**
* [Tavily Search API](https://docs.tavily.com/documentation/api-reference/endpoint/search)
* [MediaWiki Action API: search](https://www.mediawiki.org/wiki/API:Search)

**Paper (only if you want the theory behind tool-using agents)**
* Yao et al. (2022), [*ReAct: Synergizing Reasoning and Acting in Language Models*](https://arxiv.org/abs/2210.03629) — the reason → act → observe loop that most tool-calling agents build on. Our sub-agents use a fixed pipeline instead of a free loop, which is simpler and cheaper.

Here we focus on the yellow box:

In [ ]:
graph = """flowchart TD
    START([Start]) --> EXTRACT[Extract trip details]
    EXTRACT --> VALIDATE{Request complete?}

    VALIDATE -->|No| ASK[Ask traveller]
    ASK --> EXTRACT

    VALIDATE -->|Yes| FANOUT{{Send one worker per interest}}
    FANOUT --> WORKERS

    subgraph WORKERS[Parallel research sub-agents]
        direction TD
        WRITE[Write search queries] --> WEB[Search Tavily]
        WRITE --> WIKI[Search Wikipedia]
        WEB --> EXTRACT_ACT[Extract activities]
        WIKI --> EXTRACT_ACT
    end

    WORKERS --> REDUCE[(Merge activities and notes)]
    REDUCE --> END([End])
"""

showmm(graph)

### Schemas

`Activity` is what a sub-agent returns. `activities` in the parent state is a list of dicts (same reason as in M1: easy to checkpoint). We use `with_structured_output(Activity)` to force the LLM to return a structured object.

In [ ]:
import operator
from pydantic import BaseModel, Field


class Activity(BaseModel):
    name: str = Field(description="Name of the place or activity, specific enough to find on a map, e.g. 'Time Out Market Lisboa'.")
    description: str = Field(description="1-2 sentences: what you do there and why it is worth it.")
    area: str = Field(description="Neighbourhood or street, e.g. 'Cais do Sodré'.")
    duration_hours: float = Field(description="Realistic visit time in hours.")
    source_url: str = Field(description="URL of the search result or article this is based on.")
    interest: str = Field(default="", description="The interest this activity belongs to (filled in by code).")


class ActivityList(BaseModel):
    activities: list[Activity] = Field(description="At most 5 activities. Never repeat an activity.")


class SearchQueries(BaseModel):
    queries: list[str] = Field(description="2 distinct web search queries.")
    wiki_query: str = Field(description="One short Wikipedia search term, e.g. 'Lisbon architecture'.")

### State of the sub-agent

Every sub-agent has its own state `InterestState`, i.e.:
* `interest`, `destination` — the input we pass through `Send`.
* `web_results`, `wiki_results` — written by two nodes that run **in parallel**; therefore they must write to *different* keys. Otherwise, one node would overwrite the other's results.
* `activities`, `notes` — the output. They have the **same names and reducers as in the parent state**, which is how the sub-graph hands its results back.

`notes` collects non-fatal problems ("Tavily failed for X") so that one broken source never crashes the graph.

In [ ]:
class InterestState(TypedDict):
    interest: str
    destination: str
    queries: SearchQueries
    web_results: list[str]
    wiki_results: list[str]
    # operator.add is a reducer that concatenates lists. It is used here to combine the results of multiple sub-agents into a single list.
    activities: Annotated[list[dict], operator.add] 
    notes: Annotated[list[str], operator.add]

In [ ]:
# Improvement: the queries ignore the dates you provided
query_instructions = """You help plan a trip to {destination}. The traveller is interested in: {interest}.
Write 2 web search queries that find concrete places and activities (not generic blog posts) and one short Wikipedia search term."""


def write_queries(state: InterestState):
    chain = llm.with_structured_output(SearchQueries)
    queries = chain.invoke(query_instructions.format(destination=state["destination"], interest=state["interest"]))
    return {"queries": queries}

In [ ]:
# TODO: create an InterestState with a sample interest and destination, and call write_queries() to see the output.

<details>
<summary>Reference solution</summary>

``` python

interest_state = InterestState({"interest": "food and beer", "destination": "Reigoldswil, Switzerland"})
queries = write_queries(interest_state)
interest_state["queries"] = queries["queries"]
interest_state

```
</details>

### Tools: Tavily and Wikipedia

Tavily comes from `langchain-tavily`; Wikipedia is queried through its public API (it requires a `User-Agent` header). Both calls are wrapped in `try/except` — external services fail.

In [ ]:
from langchain_tavily import TavilySearch
import requests

tavily = TavilySearch(max_results=4)


def search_web(state: InterestState):
    results, notes = [], []
    for q in state["queries"].queries:
        try:
            response = tavily.invoke({"query": q})
            for r in response["results"]:
                results.append(f"URL: {r['url']}\nTITLE: {r['title']}\n{r['content'][:1000]}")
        except Exception as e:
            notes.append(f"Web search failed for '{q}' ({type(e).__name__}).")
    return {"web_results": results, "notes": notes}


HEADERS = {"User-Agent": "travel-agent-lab/0.1 (course demo)"}  # Wikipedia and Nominatim require a User-Agent


def search_wikipedia(state: InterestState):
    params = {"action": "query", "format": "json", "generator": "search", "gsrsearch": state["queries"].wiki_query,
              "gsrlimit": 2, "prop": "extracts|info", "inprop": "url", "exintro": 1, "explaintext": 1}
    try:
        r = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=HEADERS, timeout=15)
        r.raise_for_status()
        pages = r.json().get("query", {}).get("pages", {}).values()
        results = [f"URL: {p['fullurl']}\nTITLE: {p['title']}\n{p.get('extract', '')[:2500]}" for p in pages]
        return {"wiki_results": results}
    except Exception as e:
        return {"wiki_results": [], "notes": [f"Wikipedia failed for '{state['interest']}' ({type(e).__name__})."]}

In [ ]:
# TODO: use your `queries` from above to call search_web() and search_wikipedia() and see the output. You can also test with a different interest and destination.
# tav_res = search_web(interest_state)
# wiki_res = search_wikipedia(interest_state)
# interest_state.update(**wiki_res)
# interest_state.update(**tav_res)
# interest_state

### Extract activities

The LLM turns raw search text into structured `Activity` objects. It may only use places that appear in the sources
(we want the plan grounded, not hallucinated). If nothing is found we record a note instead of failing.

In [ ]:
extract_activities_instructions = """You are building a trip plan for {destination}. Interest: {interest}.
From the sources below extract 3-5 concrete visitable places or activities in or very near {destination}.
Rules:
- Return at most 5 activities, never repeat one, and keep every field short.
- Only use places mentioned in the sources. Do not invent anything.
- The name must be specific enough to look up on a map.
- source_url must be the URL of the source the place came from.

SOURCES:
{sources}"""


def extract_activities(state: InterestState):
    sources = "\n\n---\n\n".join(state.get("web_results", []) + state.get("wiki_results", []))
    if not sources:
        return {"notes": [f"No sources found for interest '{state['interest']}'."]}
    chain = llm.with_structured_output(ActivityList)
    out = chain.invoke(extract_activities_instructions.format(
        destination=state["destination"], interest=state["interest"], sources=sources))
    acts = [a.model_copy(update={"interest": state["interest"]}).model_dump() for a in out.activities[:5]]
    if not acts:
        return {"notes": [f"No activities found for interest '{state['interest']}'."]}
    return {"activities": acts}

In [ ]:
# TODO: apply extract_activities to your interest_state and see the output. You can also test with a different interest and destination.
# extract_activities(interest_state)

In [ ]:
research_builder = StateGraph(InterestState)
research_builder.add_node("write_queries", write_queries)
research_builder.add_node("search_web", search_web)
research_builder.add_node("search_wikipedia", search_wikipedia)
research_builder.add_node("extract_activities", extract_activities)

research_builder.add_edge(START, "write_queries")
research_builder.add_edge("write_queries", "search_web")
research_builder.add_edge("write_queries", "search_wikipedia")
research_builder.add_edge(["search_web", "search_wikipedia"], "extract_activities")
research_builder.add_edge("extract_activities", END)

research_graph = research_builder.compile()
display(Image(research_graph.get_graph().draw_mermaid_png()))

Try one sub-agent on its own before wiring it into the parent.

In [ ]:
test = research_graph.invoke({"interest": "street art", "destination": "Lisbon"})
for a in test["activities"]:
    print(f"- {a['name']} ({a['area']}, {a['duration_hours']}h) {a['source_url']}")
print(test["notes"])

### fan-out (`Send`)

After `validate` there are two possible futures: ask the human (problems) or research. The routing function below returns
**either** `"ask"` **or a list of `Send` objects**, one per interest. Each `Send` names the node and the *private input* for that run.

Try writing it yourself: cap the interests at `MAX_INTERESTS` so that cost stays bounded.

In [ ]:
from langgraph.types import Send

MAX_INTERESTS = 5


class PlannerState(TripState):
    activities: Annotated[list[dict], operator.add]
    notes: Annotated[list[str], operator.add]


# we use the same ask node as before, so the route_after_validate function can find it
def route_after_validate(state: PlannerState):
    if state["problems"]:
        return "ask"
    req = state["request"]
    interests = req["interests"][:MAX_INTERESTS]
    return [Send("research_interest", {"interest": i, "destination": req["destination"]}) for i in interests]

A node can also be a sub-graph, which is how we run the research sub-agent.

In [ ]:
planner_builder = StateGraph(PlannerState)
planner_builder.add_node("extract", extract)
planner_builder.add_node("validate", validate)
planner_builder.add_node("ask", ask)
planner_builder.add_node("research_interest", research_graph)  # a compiled graph can be used as a node

planner_builder.add_edge(START, "extract")
planner_builder.add_edge("extract", "validate")
planner_builder.add_conditional_edges("validate", route_after_validate, ["ask", "research_interest"])
planner_builder.add_edge("ask", "extract")
planner_builder.add_edge("research_interest", END)

planner_graph = planner_builder.compile(checkpointer=MemorySaver())
display(Image(planner_graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "planner-1"}}
first = {"messages": [HumanMessage(content="I want to go to Lisbon from the 14th to the 16th of next month. I love food, architecture and the sea.")]}

result = run_with_human(planner_graph, first, config)

print(f"{len(result['activities'])} activities found")
for a in result["activities"]:
    print(f"[{a['interest']}] {a['name']} - {a['area']}")
print("Notes:", result["notes"])

**M2 done.** Interests are researched in parallel, results are merged by the reducer and failures end up in `notes`.
Next (M3): put the activities on a map and build the day schedule.

# M3 — Geo data and the day-by-day schedule

A schedule is only good if the stops are close to each other. So before planning we

1. **geocode** every activity with OpenStreetMap's **Nominatim** (free, no key, but max **1 request/second** and a `User-Agent` is mandatory),
2. compute **distances and estimated travel times** between all places,
3. let a **planner** LLM build the schedule with *structured output* (`Schedule`), using that data.

```
research_interest (x N, parallel) -> geocode -> plan_schedule -> END
```

### 📚 Resources — M3

**LangGraph**
* [Graph API: state schemas](https://docs.langchain.com/oss/python/langgraph/graph-api) — a node's argument annotation is its **input schema**; this matters for `ScheduledState` below
* [How-to: use the Graph API](https://docs.langchain.com/oss/python/langgraph/use-graph-api) — private/input/output state
* [Structured output in LangChain](https://docs.langchain.com/oss/python/langchain/structured-output)

**Other**
* [Nominatim Search API](https://nominatim.org/release-docs/latest/api/Search/) and its [usage policy](https://operations.osmfoundation.org/policies/nominatim/) (1 request/s, User-Agent required)
* [Haversine formula](https://en.wikipedia.org/wiki/Haversine_formula) — great-circle distance between two lat/lon points

Goal:
- the `geocode` node finds the coordinates of every activity and adds them to the state and calculates pairwise distances between all activities. It also estimates travel times and adds them to the state.
- the `plan_schedule` node uses that data to build a day-by-day schedule.

In [ ]:
graph = """
    flowchart LR
    START([Start]) -->  GEOCODE[geocode] -->
    PLAN[plan_schedule] -->  END([End])
    """

showmm(graph)

### Geocoding

Results go into a **new state key `geo`** (name -> coordinates) instead of modifying `activities`, because `activities` has an
`operator.add` reducer: writing to it again would *append duplicates*. Places that cannot be found are simply left out and a note is added.

First we extend the state. **Important:** LangGraph uses the type annotation of a node's `state` argument as that node's *input schema*, so a node only sees the keys of the class it is annotated with. A node that needs `geo` must be annotated with a state class that has `geo`.

In [ ]:
class ScheduledState(PlannerState):
    geo: dict
    schedule: dict
    feedback: str

In [ ]:
import math
import time


def nominatim(query: str):
    r = requests.get("https://nominatim.openstreetmap.org/search",
                     params={"q": query, "format": "json", "limit": 1}, headers=HEADERS, timeout=15)
    time.sleep(1.1)  # Nominatim usage policy: max 1 request per second
    r.raise_for_status()
    data = r.json()
    return (float(data[0]["lat"]), float(data[0]["lon"])) if data else None


def haversine_km(a, b):
    lat1, lon1, lat2, lon2 = map(math.radians, [a[0], a[1], b[0], b[1]])
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * math.asin(math.sqrt(h))


def travel_estimate(km_straight: float):
    """Rough estimate: real routes are ~30% longer than the straight line. Walk if short, otherwise public transport/taxi."""
    km = km_straight * 1.3
    if km <= 2:
        return {"mode": "walk", "minutes": round(km / 5 * 60)}
    return {"mode": "transit/taxi", "minutes": round(km / 20 * 60 + 10)}

In [ ]:
# TODO: test the formulas for a route you know (e.g. from the campus to the nearest train station)
# this service is very shaky, e.g. try to find "Train station Brugg/Windisch, Switzerland"
# you can see the search results on: https://nominatim.openstreetmap.org/search
longlat_fhnw = nominatim("FHNW Campus Brugg Windisch, Switzerland")
longlat_sbb = nominatim("SBB Brugg, Switzerland")
km = haversine_km(longlat_fhnw, longlat_sbb)
estimate = travel_estimate(km)
print(f"Distance: {km:.2f} km, estimated travel: {estimate['mode']} in {estimate['minutes']} minutes")

In [ ]:
# our first node: remember langgraph updates the state in-place, so we can add geo data to the existing state and return it
# since ScheduleState inherits from PlannerState, it has all the fields of PlannerState plus geo, schedule and feedback.
def geocode(state: ScheduledState):
    destination = state["request"]["destination"]
    notes, geo = [], {}
    try:
        center = nominatim(destination)
    except Exception as e:
        return {"geo": {}, "notes": [f"Geocoding unavailable ({type(e).__name__}); distances could not be computed."]}
    if not center:
        return {"geo": {}, "notes": [f"Could not locate '{destination}' on the map; distances could not be computed."]}

    seen = set()
    for a in state["activities"]:
        if a["name"] in seen:
            continue
        seen.add(a["name"])
        point = None
        for q in (f"{a['name']}, {a['area']}, {destination}", f"{a['name']}, {destination}"):
            try:
                point = nominatim(q)
            except Exception:
                point = None
            if point:
                break
        if point and haversine_km(center, point) > 60:
            point = None  # probably a wrong match in another city
        if point:
            geo[a["name"]] = {"lat": point[0], "lon": point[1], "km_from_center": round(haversine_km(center, point), 1)}
        else:
            notes.append(f"Could not locate '{a['name']}' on the map (no distance data).")
    return {"geo": geo, "notes": notes}

In [ ]:
# Build a minimal ScheduledState with a few activities and call geocode() directly
geo_test_state = ScheduledState()
geo_test_state.update(result)
geo_test_state.update(geocode(geo_test_state))
print(f"Located {len(geo_test_state['geo'])} of {len(geo_test_state['activities'])} places")
for name, g in geo_test_state["geo"].items():
    print(f"- {name}: ({g['lat']:.5f}, {g['lon']:.5f}), {g['km_from_center']} km from center")
print("Notes:", geo_test_state["notes"])

### Schedule schema

Notice the `transport_from_previous` field: the planner must say *how* to get from one stop to the next.

In [ ]:
class Slot(BaseModel):
    start_time: str = Field(description="HH:MM, 24h.")
    end_time: str = Field(description="HH:MM, 24h.")
    activity: str = Field(description="Exact name of an activity from the list, or 'Lunch' / 'Dinner' / 'Free time'.")
    interest: str = Field(description="The interest it serves, or 'rest'.")
    transport_from_previous: str = Field(description="How to get here from the previous stop, e.g. 'Walk, ~10 min' or 'Tram 28, ~25 min'. Empty for the first slot of the day.")
    notes: str = Field(default="", description="Practical tip, max 1 sentence.")


class Day(BaseModel):
    date: date
    theme: str = Field(description="Short theme of the day, e.g. 'Old town and flavours'.")
    slots: list[Slot]


class Schedule(BaseModel):
    summary: str = Field(description="2-3 sentence overview of the trip.")
    days: list[Day]

### The planner

Inputs for the LLM:
- the trip dates,
- the activities (with duration and distance from the centre) and 
- a **matrix of estimated travel minutes** between geocoded places.

Rules in the prompt keep it realistic. The `feedback` part stays empty now; in M4 the critic will fill it and send the planner back for a revision.

In [ ]:
def travel_matrix(geo: dict) -> str:
    names = list(geo)
    lines = []
    for i, a in enumerate(names):
        row = []
        for j, b in enumerate(names):
            if i < j:
                t = travel_estimate(haversine_km((geo[a]["lat"], geo[a]["lon"]), (geo[b]["lat"], geo[b]["lon"])))
                row.append(f"{b}: {t['minutes']} min ({t['mode']})")
        if row:
            lines.append(f"{a} -> " + "; ".join(row))
    return "\n".join(lines) or "(no distance data available)"


In [ ]:
# call travel_matrix on geo_test_result["geo"] and see the output. It should show pairwise travel times between all located activities.
print(travel_matrix(geo_test_state["geo"]))

In [ ]:
plan_instructions = """You are an expert travel planner. Build a day-by-day schedule for {destination}.
Dates (one Day per date, exactly these): {dates}
Traveller interests: {interests}

ACTIVITIES (name | interest | area | visit hours | km from city centre):
{activities}

ESTIMATED TRAVEL MINUTES between places (straight-line based):
{matrix}

Rules:
- Use only activities from the list (plus Lunch, Dinner, Free time). You do not need to use all of them; skip the least practical ones.
- Group places that are close to each other on the same day. Avoid long trips back and forth.
- Day starts around 09:00-10:00, ends by 21:30 at the latest. Include lunch and dinner and leave buffer time.
Add the travel time from the matrix to the schedule: start_time of a slot >= end of previous + travel time. (Our code will overwrite the transport text with exact estimates, but the TIMES are your job.)
- Balance the interests over the days. Max 4 main activities per day.
- If a place is far from the centre (> 15 km), give it a dedicated half-day and mention the transport.
- Never create a slot for travelling itself (no 'Travel to ...' slots); travel time is the gap between slots.
- Never use 'Free time' blocks longer than 60 minutes. End the day after dinner instead of padding it.
- Whenever you walk, the straight-line travel minutes above must be small (< 25 min); otherwise name a vehicle.
{previous}{feedback}"""


def fix_transport(schedule: dict, geo: dict) -> dict:
    """LLMs are less reliable with numbers. The LLM chooses order and durations; code owns transport text and clock times:
    it writes our travel estimate into the slot and pushes later slots back when the gap is too short."""
    def mm(t): return int(t[:2]) * 60 + int(t[3:])
    def hhmm(m): return f"{m // 60:02d}:{m % 60:02d}"

    for day in schedule["days"]:
        # the LLM sometimes adds slots for the travel itself; the gaps between slots already cover travel
        day["slots"] = [s for s in day["slots"] if s["activity"].strip() and not s["activity"].lower().startswith("travel")]
        while day["slots"] and day["slots"][-1]["activity"] == "Free time":
            day["slots"].pop()  # no filler at the end of the day
        shift, prev_slot, last_place = 0, None, None
        for s in day["slots"]:
            start, end = mm(s["start_time"]) + shift, mm(s["end_time"]) + shift
            if prev_slot and last_place in geo and s["activity"] in geo:
                a, b = geo[last_place], geo[s["activity"]]
                est = travel_estimate(haversine_km((a["lat"], a["lon"]), (b["lat"], b["lon"])))
                label = "Walk" if est["mode"] == "walk" else "Transit/taxi"
                s["transport_from_previous"] = f"{label}, ~{est['minutes']} min"
                gap = start - mm(prev_slot["end_time"])
                if gap < est["minutes"]:
                    delta = est["minutes"] - gap
                    shift, start, end = shift + delta, start + delta, end + delta
            s["start_time"], s["end_time"] = hhmm(start), hhmm(end)
            if s["activity"] in {"Lunch", "Dinner", "Free time"}:
                s["transport_from_previous"] = ""  # only real places get a transport hint
            prev_slot = s
            if s["activity"] not in {"Lunch", "Dinner", "Free time"}:
                last_place = s["activity"]
    return schedule


def plan_schedule(state: ScheduledState):
    req = state["request"]
    start, end = date.fromisoformat(req["start_date"]), date.fromisoformat(req["end_date"])
    dates = [(start.fromordinal(start.toordinal() + i)).isoformat() for i in range((end - start).days + 1)]
    geo = state.get("geo", {})
    acts = "\n".join(
        f"{a['name']} | {a['interest']} | {a['area']} | {a['duration_hours']}h | "
        f"{geo.get(a['name'], {}).get('km_from_center', 'unknown')}"
        for a in state["activities"]
    )
    feedback = state.get("feedback", "")
    previous = ""
    if feedback and state.get("schedule"):
        previous = "\nYOUR PREVIOUS SCHEDULE (keep what works, change only what the feedback criticises):\n" + schedule_as_text(state["schedule"]) + "\n"
    if feedback:
        feedback = "\nA REVIEWER GAVE THIS FEEDBACK. Fix every point:\n" + feedback
    prompt = plan_instructions.format(
        destination=req["destination"], dates=", ".join(dates), interests=", ".join(req["interests"]),
        activities=acts, matrix=travel_matrix(geo), previous=previous, feedback=feedback)
    schedule = llm.with_structured_output(Schedule).invoke(prompt)
    return {"schedule": fix_transport(schedule.model_dump(mode="json"), geo)}

In [ ]:
plan_schedule(geo_test_state)  # test the function with the geocoded activities

### Wire it all together

We extend the state and the graph from M2. Note how `geocode` runs **once**, after *all* parallel `research_interest` branches have finished.

In [ ]:
sched_builder = StateGraph(ScheduledState)
sched_builder.add_node("extract", extract)
sched_builder.add_node("validate", validate)
sched_builder.add_node("ask", ask)
sched_builder.add_node("research_interest", research_graph)
sched_builder.add_node("geocode", geocode)
sched_builder.add_node("plan_schedule", plan_schedule)

sched_builder.add_edge(START, "extract")
sched_builder.add_edge("extract", "validate")
sched_builder.add_conditional_edges("validate", route_after_validate, ["ask", "research_interest"])
sched_builder.add_edge("ask", "extract")
sched_builder.add_edge("research_interest", "geocode")
sched_builder.add_edge("geocode", "plan_schedule")
sched_builder.add_edge("plan_schedule", END)

sched_graph = sched_builder.compile(checkpointer=MemorySaver())
display(Image(sched_graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "sched-1"}}
first = {"messages": [HumanMessage(content="I want to go to Lisbon from the 14th to the 16th of next month. I love food, architecture and the sea.")]}

result = run_with_human(sched_graph, first, config)

print(f"Located {len(result['geo'])} of {len({a['name'] for a in result['activities']})} places")
print("Notes:", result["notes"])
print()
sched = Schedule(**result["schedule"])
print(sched.summary)
for day in sched.days:
    print(f"\n== {day.date}: {day.theme}")
    for s in day.slots:
        print(f"  {s.start_time}-{s.end_time}  {s.activity}   [{s.transport_from_previous}]")

### Quality check (plain Python, no LLM)

The schedule is valid but not necessarily *good*. This cell flags three typical problems; in M4 a critic agent will find these (and more) by itself. The flagged problems are: long "Free time" blocks, places with no coordinates, and a gap between two stops that is shorter than the estimated travel time.

In [ ]:
def to_min(t: str) -> int:
    h, m = t.split(":")
    return int(h) * 60 + int(m)


def quality_issues(sched: Schedule, geo: dict) -> list[str]:
    issues, filler = [], {"Lunch", "Dinner", "Free time"}
    for day in sched.days:
        prev = None  # last real place of the day
        prev_end = None
        for s in day.slots:
            if s.activity == "Free time" and to_min(s.end_time) - to_min(s.start_time) > 90:
                issues.append(f"{day.date}: {to_min(s.end_time) - to_min(s.start_time)} min of 'Free time' ({s.start_time}-{s.end_time})")
            if s.activity in filler:
                prev_end = to_min(s.end_time)
                continue
            if s.activity not in geo:
                issues.append(f"{day.date}: '{s.activity}' has no coordinates, so its distances were never checked")
            elif prev in geo:
                a, b = geo[prev], geo[s.activity]
                est = travel_estimate(haversine_km((a["lat"], a["lon"]), (b["lat"], b["lon"])))
                if prev_end is not None and to_min(s.start_time) - prev_end < est["minutes"] - 5:
                    issues.append(f"{day.date}: only {to_min(s.start_time) - prev_end} min between '{prev}' and '{s.activity}', but travel takes ~{est['minutes']} min")
            prev = s.activity
            prev_end = to_min(s.end_time)
        if prev_end and prev_end > 22 * 60:
            issues.append(f"{day.date}: the day ends at {day.slots[-1].end_time}, which is very late")
    return issues


issues = quality_issues(sched, result["geo"])
print(f"{len(issues)} issue(s) found" if issues else "No issues found")
for i in issues:
    print(" -", i)

**M3 done.** We have a first schedule with transport hints. Next (M4): a critic agent reviews it, the planner revises (max 2 rounds), and the human approves.

# M4 — The critic loop and human approval

A first draft is rarely good. We add two things:

1. A **critic agent** that reviews the schedule (is the pace too much? is it feasible? is the transport between spots realistic?) and returns a structured `Critique`.
   If it finds problems, the graph **loops back** to `plan_schedule` with the feedback. Maximum **2 revision rounds**, so cost stays bounded.
2. A **human approval** step (`interrupt()` again): the human can approve, or type feedback that sends the planner back for another round.



In [ ]:
graph = """flowchart TD
    START([Start]) --> PLAN[plan_schedule]
    PLAN --> CRITIQUE[critique]
    CRITIQUE --> ROUTE{Passed or max revisions?}

    ROUTE -->|No, rounds left| PLAN
    ROUTE -->|Yes| APPROVE[approve]

    APPROVE --> PAUSE[[Wait for human response]]
    PAUSE --> DECISION{Approved?}

    DECISION -->|Feedback, reset revisions| PLAN
    DECISION -->|Approved| END([End])
"""

showmm(graph)

### 📚 Resources — M4

**LangGraph**
* [Graph API: conditional edges](https://docs.langchain.com/oss/python/langgraph/graph-api) — `add_conditional_edges` for the revise-or-approve loop
* [Workflows and agents: evaluator-optimizer](https://docs.langchain.com/oss/python/langgraph/workflows-agents) — the generate → critique → revise pattern we implement
* [Interrupts (human-in-the-loop)](https://docs.langchain.com/oss/python/langgraph/interrupts) — the approval step
* [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) — threads and state history

**Paper**
* Madaan et al. (2023), [*Self-Refine: Iterative Refinement with Self-Feedback*](https://arxiv.org/abs/2303.17651) — the generate → critique → revise loop. More papers on critic choice and reasoning are listed in Exercise 3a.



So far **one** model (`gpt-4o-mini`) plans *and* critiques. Is that wise? 

1. **Self-preference:** the planner and the critic are the same model. What could go wrong when a model grades its own output? Would you expect it to be more or less strict than a different model?
2. **Cost and latency:** the planner runs up to ~7 times per trip, the critic only a few times, but it reads the whole schedule plus the travel matrix. Where is a more expensive model worth the money?
3. **Task difficulty:** the critic has to *compare numbers* (gap between slots vs. travel minutes), *count* (activities per day) and *judge* (is the pace realistic?). Which of these do small non-reasoning models do badly? What does a reasoning model do before it answers that a plain model does not?
4. **Failure modes:** a bad critic is either too lenient (bad plans pass) or too strict (the loop never converges and burns tokens). Look at `critiques` after a run: is yours hallucinating problems or missing real ones?
5. **Experiment:** run the notebook once with `CRITIC_MODEL = "gpt-4o-mini"` and once with a reasoning model. Compare the number of critic rounds, the number of issues that are *real* (check them against `quality_issues`), and the runtime.


#### Further reading
* Huang et al. (2023), [Large Language Models Cannot Self-Correct Reasoning Yet](https://arxiv.org/abs/2310.01798): without external feedback, self-critique often does not help and can make answers worse.
* Kamoi et al. (2024), [When Can LLMs Actually Correct Their Own Mistakes?](https://arxiv.org/abs/2406.01297): survey. Self-correction works when reliable external feedback exists (like our plain-Python `quality_issues`), not from the model alone.
* Panickssery et al. (2024), [LLM Evaluators Recognize and Favor Their Own Generations](https://arxiv.org/abs/2404.13076) and Wataoka et al. (2024), [Self-Preference Bias in LLM-as-a-Judge](https://arxiv.org/abs/2410.21819): models rate their own output higher. This is the argument for a *different* critic model.
* Zheng et al. (2023), [Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685): describes "self-enhancement bias" and other judge biases.
* McAleese et al. (2024), [LLM Critics Help Catch LLM Bugs](https://arxiv.org/abs/2407.00215): trained critics catch more errors than human reviewers alone.
* Snell et al. (2024), [Scaling LLM Test-Time Compute Optimally can be More Effective than Scaling Model Parameters](https://arxiv.org/abs/2408.03314): spending more compute at inference (thinking longer) can beat a larger model. This is the argument for *reasoning* in the critic.

(Note that several papers study correctness tasks like maths and code; a travel schedule is softer, so treat them as guidance and measure it yourself in step 5.)

In [ ]:
# Planner/extractor/researchers keep the cheap model. Only the critic changes.
# gpt-5-mini is a reasoning model: it takes reasoning_effort
# Set CRITIC_MODEL = "gpt-4o-mini" to reproduce the "same model" baseline.
CRITIC_MODEL = "gpt-5-mini"
CRITIC_REASONING_EFFORT = "medium"  # "minimal" | "low" | "medium" | "high"

if CRITIC_MODEL.startswith(("gpt-5", "o1", "o3", "o4")):
    critic_llm = ChatOpenAI(model=CRITIC_MODEL, reasoning_effort=CRITIC_REASONING_EFFORT)
else:
    critic_llm = ChatOpenAI(model=CRITIC_MODEL, temperature=0)

### Critique schema

The critic does not just say yes/no: separate verdicts per aspect force it to actually look at each one.

In [ ]:
class Critique(BaseModel):
    pace_ok: bool = Field(description="False if days are overloaded, have too little buffer, or contain long idle filler blocks.")
    feasibility_ok: bool = Field(description="False if opening-hour/time logic or the visit durations are unrealistic.")
    transport_ok: bool = Field(description="False if movements between places are not credible (e.g. walking far, ignoring travel time, far-away places without planning).")
    issues: list[str] = Field(description="Concrete problems, each one citing the day and activity. Empty if everything is fine.")
    suggestions: list[str] = Field(description="Concrete fixes for the planner, one per issue.")

    @property
    def passed(self) -> bool:
        return self.pace_ok and self.feasibility_ok and self.transport_ok

class CriticState(ScheduledState):
    critiques: Annotated[list[dict], operator.add]
    revisions: int
    approved: bool

The prompt decides how good the critic is. Read the one below, then improve it: e.g. add rules about museum closing days,
a maximum of walking minutes per day, or kids/seniors. We also hand the critic the findings of our **plain-Python quality check from M3**,
so it combines hard facts with judgement.

In [ ]:
critic_instructions = """You are a strict but fair travel-plan critic. Review this schedule for a trip to {destination}.

SCHEDULE:
{schedule}

ESTIMATED TRAVEL MINUTES between places (straight-line based):
{matrix}

AUTOMATIC CHECK FINDINGS:
{auto_findings}

Judge three aspects:
1. PACE: Is it too much (more than ~4 main activities, no breaks) or too little (hours of pure 'Free time')?
2. FEASIBILITY: Are visit durations and times realistic (meals at normal hours, nothing after 21:30)? Is a far-away place squeezed in?
3. TRANSPORT: Is the transport hint between consecutive stops credible? Each slot shows the gap since the previous slot: the gap must cover the travel time in the transport hint. Do not compare against the matrix for non-consecutive places.

Cite the day and activity for every issue and give one concrete fix per issue.
Be pragmatic: only report SERIOUS problems. A transport time that is off by less than 15 minutes, or a missing buffer of a few minutes, is NOT an issue. If it is a good plan, pass it with no issues.
Ignore findings about missing coordinates: the planner cannot fix them."""


def schedule_as_text(schedule: dict) -> str:
    lines = [schedule["summary"]]
    for day in schedule["days"]:
        lines.append(f"\n{day['date']} - {day['theme']}")
        prev_end = None
        for s in day["slots"]:
            via = f" [{s['transport_from_previous']}]" if s["transport_from_previous"] else ""
            start = to_min(s["start_time"])
            gap = f" (gap since previous slot: {start - prev_end} min)" if prev_end is not None else ""
            lines.append(f"  {s['start_time']}-{s['end_time']}  {s['activity']}{via}{gap}")
            prev_end = to_min(s["end_time"])
    return "\n".join(lines)


MAX_REVISIONS = 2


def critique(state: CriticState):
    sched = Schedule(**state["schedule"])
    auto = [i for i in quality_issues(sched, state.get("geo", {})) if "no coordinates" not in i]
    prompt = critic_instructions.format(
        destination=state["request"]["destination"], schedule=schedule_as_text(state["schedule"]),
        matrix=travel_matrix(state.get("geo", {})), auto_findings="\n".join(f"- {i}" for i in auto) or "none")
    result = critic_llm.with_structured_output(Critique).invoke(prompt)
    feedback = ""
    if not result.passed:
        feedback = "\n".join(f"- {i}" for i in result.issues) + "\nFixes:\n" + "\n".join(f"- {s}" for s in result.suggestions)
    return {
        "critiques": [{**result.model_dump(), "passed": result.passed}],
        "feedback": feedback,
        "revisions": state.get("revisions", 0) + (0 if result.passed else 1),
    }


def route_after_critique(state: CriticState):
    if state["critiques"][-1]["passed"] or state["revisions"] > MAX_REVISIONS:
        return "approve"
    return "plan_schedule"

### Human approval

`approve` shows the schedule and waits. If the answer is an approval word, we are done. Anything else is treated as **feedback** for the planner.
The loop counter is reset so that the critic gets its two rounds again on the human's feedback.

In [ ]:
APPROVE_WORDS = {"approve", "approved", "ok", "okay", "yes", "y", "good", "looks good", "lgtm"}


def approve(state: CriticState):
    notes = state.get("critiques", [])[-1]
    status = "The critic approved this plan." if notes["passed"] else \
        f"The critic still has concerns after {MAX_REVISIONS} revisions:\n{state['feedback']}"
    answer = interrupt(f"{schedule_as_text(state['schedule'])}\n\n{status}\n\nApprove, or tell me what to change.")
    if answer.strip().lower().rstrip("!. ") in APPROVE_WORDS:
        return {"approved": True}
    return {"approved": False, "feedback": f"- The traveller asks: {answer}", "revisions": 0}


def route_after_approve(state: CriticState):
    return END if state["approved"] else "plan_schedule"

In [ ]:
crit_builder = StateGraph(CriticState)
crit_builder.add_node("extract", extract)
crit_builder.add_node("validate", validate)
crit_builder.add_node("ask", ask)
crit_builder.add_node("research_interest", research_graph)
crit_builder.add_node("geocode", geocode)
crit_builder.add_node("plan_schedule", plan_schedule)
crit_builder.add_node("critique", critique)
crit_builder.add_node("approve", approve)

crit_builder.add_edge(START, "extract")
crit_builder.add_edge("extract", "validate")
crit_builder.add_conditional_edges("validate", route_after_validate, ["ask", "research_interest"])
crit_builder.add_edge("ask", "extract")
crit_builder.add_edge("research_interest", "geocode")
crit_builder.add_edge("geocode", "plan_schedule")
crit_builder.add_edge("plan_schedule", "critique")
crit_builder.add_conditional_edges("critique", route_after_critique, ["plan_schedule", "approve"])
crit_builder.add_conditional_edges("approve", route_after_approve, ["plan_schedule", END])

crit_graph = crit_builder.compile(checkpointer=MemorySaver())
display(Image(crit_graph.get_graph(xray=1).draw_mermaid_png()))

Run it. The first human answer is only needed if the request is incomplete; the **last** question is the approval.
Try typing real feedback (e.g. *"less walking, I have a bad knee"*) and watch the planner revise.

In [ ]:
# this takes a long time to run, maybe shorten the time or reduce the number of interests to speed it up
config = {"configurable": {"thread_id": "crit-1"}}
first = {"messages": [HumanMessage(content="I want to go to Lisbon from the 14th to the 16th of next month. I love food, architecture and the sea.")]}

result = run_with_human(crit_graph, first, config, scripted_answers=["Fewer long free-time blocks please", "approve"])

### What did the critic say?

Every critic round is stored in `critiques` (it has an `operator.add` reducer, so rounds accumulate). This is how you can *see* the improvement.

In [ ]:
for n, c in enumerate(result["critiques"], 1):
    verdict = "PASSED" if c["passed"] else "NEEDS WORK"
    print(f"--- Critic round {n}: {verdict}  (pace={c['pace_ok']}, feasibility={c['feasibility_ok']}, transport={c['transport_ok']})")
    for i in c["issues"]:
        print("   -", i)

print("\nFinal approved:", result["approved"])
final = Schedule(**result["schedule"])
print("Remaining automatic findings:", quality_issues(final, result["geo"]) or "none")

**M4 done.** The plan is critiqued, revised and approved by a human. Next (M5): render it as a well structured `travel_plan.html`.

# M5 — The final artefact: a structured HTML travel plan

The last node turns the approved state into **`travel_plan.html`**. This node needs **no LLM**: all the content already exists as structured data
(that is the benefit of structured output!), so we only *render* it. A template engine (**Jinja2**) keeps layout and data separate.

```
approve --(approved)--> render_html -> END
```

The page contains:
1. **Header**: destination, dates, interests
2. **Overview**: the planner's summary
3. **One card per day**: a timeline with times, activities, the transport from the previous stop, tips, a map link and the source of each activity
4. **Notes & limitations**: unresolved critic concerns, places we could not locate, failed sources. The plan is honest about what it does not know.
5. **Sources**: every URL the activities are based on

**Security note:** the content comes from web pages and an LLM, so Jinja's *autoescape* must be on. We test this below.

In [ ]:
m5_graph = """flowchart TD
    START([Start]) --> APPROVE
    APPROVE{Human approval}
    APPROVE -->|Approved| RENDER[render_html]
    RENDER --> FILE[travel_plan.html]
    FILE --> END([End])
"""

showmm(m5_graph)

### 📚 Resources — M5

**LangGraph**
* [Graph API: nodes, edges and `END`](https://docs.langchain.com/oss/python/langgraph/graph-api) — a node does not have to call an LLM
* [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview) — compile, invoke, stream

**Other**
* [Jinja2 template designer documentation](https://jinja.palletsprojects.com/en/stable/templates/) — loops, filters, autoescaping
* [Pydantic: serialization (`model_dump`)](https://docs.pydantic.dev/latest/concepts/serialization/) — turning structured output into plain dicts for the template

### Step 1 — Prepare the data (plain Python)

The template should stay simple, so we do the data work first: join each slot with its `Activity` (description, area, source) and `geo` (map link), and format the dates.

In [ ]:
from urllib.parse import quote_plus

FILLER = {"Lunch", "Dinner", "Free time"}


def build_context(state: dict) -> dict:
    req = state["request"]
    acts = {a["name"]: a for a in state["activities"]}
    geo = state.get("geo", {})

    days = []
    for d in state["schedule"]["days"]:
        d_date = date.fromisoformat(d["date"])
        slots = []
        for s in d["slots"]:
            a = acts.get(s["activity"])
            slots.append({
                **s,
                "is_filler": s["activity"] in FILLER,
                "description": a["description"] if a else "",
                "area": a["area"] if a else "",
                "source_url": a["source_url"] if a else "",
                "map_url": (f"https://www.openstreetmap.org/?mlat={geo[s['activity']]['lat']}&mlon={geo[s['activity']]['lon']}#map=16/{geo[s['activity']]['lat']}/{geo[s['activity']]['lon']}"
                            if s["activity"] in geo else ""),
            })
        days.append({"label": d_date.strftime("%A, %d %B %Y"), "theme": d["theme"], "slots": slots})

    last = state["critiques"][-1] if state.get("critiques") else None
    used = {s["activity"] for d in state["schedule"]["days"] for s in d["slots"]}
    sources = sorted({acts[n]["source_url"] for n in used if n in acts and acts[n]["source_url"]})
    notes = list(dict.fromkeys(state.get("notes", [])))  # unique, keep order

    start, end = date.fromisoformat(req["start_date"]), date.fromisoformat(req["end_date"])
    return {
        "destination": req["destination"],
        "date_range": f"{start.strftime('%d %B %Y')} – {end.strftime('%d %B %Y')}",
        "n_days": len(days),
        "interests": req["interests"],
        "summary": state["schedule"]["summary"],
        "days": days,
        "critic_passed": bool(last and last["passed"]),
        "critic_rounds": len(state.get("critiques", [])),
        "open_issues": [] if (last is None or last["passed"]) else last["issues"],
        "notes": notes,
        "sources": sources,
        "generated": date.today().strftime("%d %B %Y"),
    }

### Step 2 — The template

Read it once: `{{ ... }}` prints a value, `{% for %}` / `{% if %}` are logic, and `|e` is not needed because autoescape is on.

In [ ]:
from jinja2 import Environment, select_autoescape

TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Travel plan: {{ destination }}</title>
<style>
  :root { --ink:#1f2933; --muted:#667085; --line:#e4e7ec; --brand:#0b6e99; --bg:#f6f8fa; --warn:#fff4e5; --warn-line:#f5b971; }
  * { box-sizing: border-box; }
  body { font-family: system-ui, -apple-system, "Segoe UI", sans-serif; color: var(--ink); background: var(--bg); margin: 0; line-height: 1.5; }
  header { background: var(--brand); color: #fff; padding: 2.5rem 1.5rem; }
  header .inner, main { max-width: 900px; margin: 0 auto; }
  header h1 { margin: 0 0 .25rem; font-size: 2rem; }
  header p { margin: .25rem 0; opacity: .95; }
  .chip { display: inline-block; background: rgba(255,255,255,.2); border-radius: 999px; padding: .1rem .7rem; margin: .2rem .3rem 0 0; font-size: .85rem; }
  main { padding: 1.5rem; }
  nav { margin-bottom: 1rem; }
  nav a { margin-right: 1rem; color: var(--brand); text-decoration: none; font-weight: 600; }
  section, .day { background: #fff; border: 1px solid var(--line); border-radius: 12px; padding: 1.25rem 1.5rem; margin-bottom: 1.25rem; }
  h2 { margin-top: 0; font-size: 1.3rem; }
  .theme { color: var(--muted); margin-top: -.5rem; }
  .slot { display: grid; grid-template-columns: 5.5rem 1fr; gap: 0 1rem; padding: .75rem 0; border-top: 1px solid var(--line); }
  .slot:first-of-type { border-top: 0; }
  .time { font-variant-numeric: tabular-nums; font-weight: 600; color: var(--brand); }
  .transport { grid-column: 2; font-size: .85rem; color: var(--muted); margin-bottom: .25rem; }
  .name { font-weight: 600; }
  .filler .name { font-weight: 500; color: var(--muted); }
  .meta { font-size: .85rem; color: var(--muted); }
  .tag { display: inline-block; font-size: .75rem; background: #e8f3f8; color: var(--brand); border-radius: 6px; padding: 0 .45rem; margin-left: .4rem; }
  .tip { font-size: .9rem; }
  a { color: var(--brand); }
  .limits { background: var(--warn); border-color: var(--warn-line); }
  .ok { color: #067647; font-weight: 600; }
  ul { padding-left: 1.2rem; }
  footer { text-align: center; color: var(--muted); font-size: .8rem; padding: 1rem 0 2rem; }
  @media print { header { background: none; color: #000; } .chip { border: 1px solid #999; } nav { display: none; } .day { break-inside: avoid; } }
</style>
</head>
<body>
<header><div class="inner">
  <h1>{{ destination }}</h1>
  <p>{{ date_range }} · {{ n_days }} day{{ "s" if n_days != 1 }}</p>
  <p>{% for i in interests %}<span class="chip">{{ i }}</span>{% endfor %}</p>
</div></header>
<main>
  <nav>
    {% for d in days %}<a href="#day{{ loop.index }}">Day {{ loop.index }}</a>{% endfor %}
    <a href="#notes">Notes</a><a href="#sources">Sources</a>
  </nav>

  <section id="overview"><h2>Overview</h2><p>{{ summary }}</p></section>

  {% for d in days %}
  <div class="day" id="day{{ loop.index }}">
    <h2>Day {{ loop.index }} · {{ d.label }}</h2>
    <p class="theme">{{ d.theme }}</p>
    {% for s in d.slots %}
    <div class="slot{{ ' filler' if s.is_filler }}">
      <div class="time">{{ s.start_time }}–{{ s.end_time }}</div>
      <div>
        {% if s.transport_from_previous %}<div class="transport">↳ {{ s.transport_from_previous }}</div>{% endif %}
        <div><span class="name">{{ s.activity }}</span>{% if s.interest and s.interest != "rest" and not s.is_filler %}<span class="tag">{{ s.interest }}</span>{% endif %}</div>
        {% if s.description %}<div>{{ s.description }}</div>{% endif %}
        {% if s.notes %}<div class="tip">💡 {{ s.notes }}</div>{% endif %}
        <div class="meta">
          {% if s.area %}📍 {{ s.area }}{% endif %}
          {% if s.map_url %} · <a href="{{ s.map_url }}" target="_blank" rel="noopener noreferrer">map</a>{% endif %}
          {% if s.source_url %} · <a href="{{ s.source_url }}" target="_blank" rel="noopener noreferrer">source</a>{% endif %}
        </div>
      </div>
    </div>
    {% endfor %}
  </div>
  {% endfor %}

  <section id="notes" class="limits">
    <h2>Notes and known limitations</h2>
    {% if critic_passed %}
      <p class="ok">✓ An AI reviewer checked pace, feasibility and transport and found no serious problems ({{ critic_rounds }} review round{{ "s" if critic_rounds != 1 }}).</p>
    {% else %}
      <p>The AI reviewer still had concerns after the revision limit:</p>
      <ul>{% for i in open_issues %}<li>{{ i }}</li>{% endfor %}</ul>
    {% endif %}
    {% if notes %}<ul>{% for n in notes %}<li>{{ n }}</li>{% endfor %}</ul>{% endif %}
    <ul>
      <li>Travel times are rough estimates based on straight-line distances, not live timetables.</li>
      <li>Opening hours and prices are not verified. Check them before you go.</li>
    </ul>
  </section>

  <section id="sources">
    <h2>Sources</h2>
    <ul>{% for u in sources %}<li><a href="{{ u }}" target="_blank" rel="noopener noreferrer">{{ u }}</a></li>{% endfor %}</ul>
  </section>
</main>
<footer>Generated on {{ generated }} by an AI travel agent built with LangGraph.</footer>
</body>
</html>
"""

jinja_env = Environment(autoescape=select_autoescape(default=True, default_for_string=True), trim_blocks=True, lstrip_blocks=True)
template = jinja_env.from_string(TEMPLATE)


def render_html_string(state: dict) -> str:
    return template.render(**build_context(state))

### Step 3 — Unit checks (no LLM, no network)

Because rendering is pure Python, we can test it with hand-made data. This also checks that malicious text coming from a web page cannot inject HTML or scripts.

In [ ]:
fake_state = {
    "request": {"destination": "Testville <b>", "start_date": "2030-05-01", "end_date": "2030-05-02", "interests": ["food", "<script>alert(1)</script>"]},
    "activities": [{"name": "Café <img src=x onerror=alert(1)>", "description": "Great coffee.", "area": "Old town", "duration_hours": 1.0,
                    "source_url": "https://example.com/a?x=1&y=2", "interest": "food"}],
    "geo": {"Café <img src=x onerror=alert(1)>": {"lat": 1.5, "lon": 2.5, "km_from_center": 0.3}},
    "schedule": {"summary": "Short trip.", "days": [
        {"date": "2030-05-01", "theme": "Coffee", "slots": [
            {"start_time": "09:00", "end_time": "10:00", "activity": "Café <img src=x onerror=alert(1)>", "interest": "food", "transport_from_previous": "", "notes": "Try the cake"},
            {"start_time": "10:00", "end_time": "11:00", "activity": "Free time", "interest": "rest", "transport_from_previous": "", "notes": ""}]},
        {"date": "2030-05-02", "theme": "Rest", "slots": []}]},
    "critiques": [{"passed": False, "issues": ["Day 1 is thin"], "suggestions": [], "pace_ok": True, "feasibility_ok": True, "transport_ok": True}],
    "notes": ["Wikipedia failed for 'x'.", "Wikipedia failed for 'x'."],
}

html_out = render_html_string(fake_state)

assert "<script>alert(1)</script>" not in html_out and "&lt;script&gt;" in html_out, "interests must be escaped"
assert "<img src=x" not in html_out, "activity names must be escaped"
assert html_out.count('class="day"') == 2, "one card per day"
assert "Day 1 is thin" in html_out, "open critic issues must be shown"
assert html_out.count("Wikipedia failed") == 1, "notes are de-duplicated"
assert "openstreetmap.org/?mlat=1.5" in html_out and "x=1&amp;y=2" in html_out
assert "Wednesday, 01 May 2030" in html_out
print("✅ All rendering checks passed")

### Make the page yours

Open the generated file in a browser first, then change the template or `build_context`. Ideas, easiest first:
* show the **total travel minutes per day** (sum the numbers from `transport_from_previous`),
* add a **"Packing tips"** section: this needs one more LLM node with structured output,
* make the days **collapsible** with `<details>`,
* group the sources by interest.

### The final node and graph

`render_html` reads the final state, renders the page and writes the file. The `approve` node now leads here instead of directly to `END`.

In [ ]:
from pathlib import Path


class FinalState(CriticState):
    html_path: str


def render_html(state: FinalState):
    path = Path("travel_plan.html").resolve()
    path.write_text(render_html_string(state), encoding="utf-8")
    return {"html_path": str(path)}


def route_after_approve(state: FinalState):
    return "render_html" if state["approved"] else "plan_schedule"


final_builder = StateGraph(FinalState)
final_builder.add_node("extract", extract)
final_builder.add_node("validate", validate)
final_builder.add_node("ask", ask)
final_builder.add_node("research_interest", research_graph)
final_builder.add_node("geocode", geocode)
final_builder.add_node("plan_schedule", plan_schedule)
final_builder.add_node("critique", critique)
final_builder.add_node("approve", approve)
final_builder.add_node("render_html", render_html)

final_builder.add_edge(START, "extract")
final_builder.add_edge("extract", "validate")
final_builder.add_conditional_edges("validate", route_after_validate, ["ask", "research_interest"])
final_builder.add_edge("ask", "extract")
final_builder.add_edge("research_interest", "geocode")
final_builder.add_edge("geocode", "plan_schedule")
final_builder.add_edge("plan_schedule", "critique")
final_builder.add_conditional_edges("critique", route_after_critique, ["plan_schedule", "approve"])
final_builder.add_conditional_edges("approve", route_after_approve, ["plan_schedule", "render_html"])
final_builder.add_edge("render_html", END)

travel_agent = final_builder.compile(checkpointer=MemorySaver())
display(Image(travel_agent.get_graph(xray=1).draw_mermaid_png()))

### Run the complete travel agent

Type your own wishes, answer the follow-up questions, and approve (or give feedback on) the plan at the end.

In [ ]:
config = {"configurable": {"thread_id": "final-1"}}
wishes = "I want to go to Lisbon from the 14th to the 15th of next month. I love food and architecture."
# Replace `wishes` with input("Where do you want to go? ") to try it interactively, and drop scripted_answers.
result = run_with_human(travel_agent, {"messages": [HumanMessage(content=wishes)]}, config, scripted_answers=["approve"])
print("\nSaved:", result["html_path"])

## Recap

| Concept | Where |
|---|---|
| Structured output | `TripRequest`, `ActivityList`, `Schedule`, `Critique` |
| Human-in-the-loop | `ask` (missing info), `approve` (final say) |
| Map-reduce with `Send()` | one research sub-graph per interest |
| Sub-graphs and reducers | `research_graph`, `operator.add` on `activities`, `notes`, `critiques` |
| Cycles / reflection | `plan_schedule` ⇄ `critique` |
| Deterministic code next to LLMs | `validate`, `geocode`, `fix_transport`, `quality_issues`, `render_html` |
| Checkpointer | `MemorySaver` makes `interrupt()` possible |

**Where to go next:**
- there is a lot of room for improvements, e.g. there is repetition in the schedule (i.e. same activities at different days)
- at the moment, there is no persistent memory, see e.g. here https://docs.langchain.com/oss/python/deepagents/memory#memory
- to make the agent flow feel more natural, we could also use streaming (https://docs.langchain.com/oss/python/langgraph/streaming)
- there is also the possibility of deploying the graph in LangSmith Studio, see here https://www.youtube.com/watch?v=pLPJoFvq4_M and https://www.langchain.com/langsmith/observability